## Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

import random
from PIL import Image
import json

import numpy as np
import matplotlib.pyplot as plt

import torch
import torchvision

from utils.utils import imagenet_transform, inverse_transform

def choose_device() -> str:
    if torch.cuda.is_available():
        return "cuda:0"
    if hasattr(torch.backends, "mps"):
        if torch.backends.mps.is_available():
            return "mps"
    return "cpu"

device = torch.device(choose_device())
print(device)

# fix seed for reproducibility
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed) # multi-GPU

In [ ]:
# Load labels from json file
with open("imagenet_class_index.json") as f:
    imagenet_labels = list(json.load(f).values())

Load pre-trained ResNet50 model

In [ ]:
resnet = torchvision.models.resnet50(weights="ResNet50_Weights.IMAGENET1K_V2")
resnet.eval().to(device);

In [ ]:
img = Image.open("picture.jpeg")


### Saliency maps for top N by class predictions with LRP

In [ ]:
from zennit.canonizers import SequentialMergeBatchNorm
from zennit.composites import EpsilonGammaBox, EpsilonPlus, EpsilonAlpha2Beta1Flat, EpsilonPlusFlat, EpsilonAlpha2Beta1, GuidedBackprop

In [ ]:
img_tensor = imagenet_transform(img).unsqueeze(0).to(device)
img_tensor.requires_grad = True
img_tensor.grad = None

# get predictions
with torch.no_grad():
    preds = resnet(img_tensor)
    topN = torch.topk(preds, 3).indices.squeeze().tolist()


for pred in topN:
    print(pred, imagenet_labels[pred])

    img_tensor = imagenet_transform(img).unsqueeze(0).to(device)
    img_tensor.requires_grad = True
    img_tensor.grad = None
    with EpsilonPlus(canonizers=[SequentialMergeBatchNorm()]).context(resnet) as modified_model:
    # with EpsilonGammaBox(low=-1, high=1, canonizers=[SequentialMergeBatchNorm()]).context(resnet) as modified_model:
        output = modified_model(img_tensor)
        # gradient/ relevance wrt. class/output 0
        output[0, pred].backward()


    heatmap = img_tensor.grad.squeeze(0).cpu().detach().mean(dim=0).numpy()
    # only positive values
    heatmap = np.maximum(heatmap, 0)

    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(inverse_transform(img_tensor[0]).permute(1, 2, 0).cpu().detach().numpy())
    ax[0].set_title(f"Prediction: {pred}")
    ax[0].axis("off")


    ax[1].imshow(inverse_transform(img_tensor[0]).permute(1, 2, 0).cpu().detach().numpy())
    ax[1].imshow(heatmap, cmap="viridis", interpolation="nearest", alpha=0.6)
    ax[1].set_title("Image + Heatmap")
    ax[1].axis("off")

    ax[2].imshow(heatmap, cmap="hot", interpolation="nearest")
    ax[2].set_title("Heatmap")
    ax[2].axis("off")
    plt.tight_layout()

## Investigate intermediate relevance

A hook for storing intermediate relevance values

In [ ]:
def store_hook(module, input, output):
    if type(output) is tuple:
        output = output[-1]
    # set the current module's attribute 'output' to the its tensor
    module.output = output
    # keep the output tensor gradient, even if it is not a leaf-tensor
    output.retain_grad()

Use the whole relevance from the previous example

In [ ]:
pred = topN[0]

grads = {}
# with EpsilonGammaBox(low=-1, high=1, canonizers=[SequentialMergeBatchNorm()]).context(my_model) as modified_model:
with EpsilonPlus(canonizers=[SequentialMergeBatchNorm()]).context(resnet) as modified_model:
        handles = [
            module.register_forward_hook(store_hook) for module in modified_model.modules()
        ]

        img_tensor = imagenet_transform(img).unsqueeze(0).to(device)
        img_tensor.requires_grad = True
        output = modified_model(img_tensor)
        output[0,pred].backward()
        # activations.mean(dim=(2, 3)).view(2048)[i].backward()
        for name, module in modified_model.named_modules():
            grads[name] = module.output.grad

for handle in handles:
    handle.remove()


List layer names with intermediate relevance

In [ ]:
grads.keys()

Try different layers

In [ ]:
layer_names = (["layer1", "layer2", "layer3", "layer4"]+
               ["layer1.0", "layer2.0", "layer3.0", "layer4.0" ]+
               ["layer1.0.conv1", "layer2.0.conv1", "layer3.0.conv1", "layer4.0.conv1" ]+
               ["layer1.0.conv2", "layer2.0.conv2", "layer3.0.conv2", "layer4.0.conv2" ]+
               ["layer1.0.conv3", "layer2.0.conv3", "layer3.0.conv3", "layer4.0.conv3" ]
               )

# plot grid of heatmaps
fig, ax = plt.subplots(5, 4, figsize=(20, 20))
for i, layer_name in enumerate(layer_names):
    heatmap = grads[layer_name].squeeze(0).cpu().detach().mean(dim=0).numpy()
    heatmap = np.maximum(heatmap, 0)
    ax[i // 4, i % 4].imshow(heatmap, cmap="hot", interpolation="nearest")
    ax[i // 4, i % 4].set_title(f"{layer_name}")
    ax[i // 4, i % 4].axis("off")
    
plt.tight_layout()

In [ ]:
layer_names = (
    ["layer1.0", "layer1.1", "layer1.2"]+
    ["layer1.0.conv1", "layer1.1.conv1", "layer1.2.conv1"]+
    ["layer1.0.conv2", "layer1.1.conv2", "layer1.2.conv2"]+
    ["layer1.0.conv3", "layer1.1.conv3", "layer1.2.conv3"]
)


# plot grid of heatmaps
fig, ax = plt.subplots(4, 3, figsize=(20, 20))
for i, layer_name in enumerate(layer_names):
    heatmap = grads[layer_name].squeeze(0).cpu().detach().mean(dim=0).numpy()
    heatmap = np.maximum(heatmap, 0)
    ax[i // 3, i % 3].imshow(heatmap, cmap="hot", interpolation="nearest")
    ax[i // 3, i % 3].set_title(layer_name)
    ax[i // 3, i % 3].axis("off")
plt.tight_layout()

In [ ]:
layer_names = (
    ["layer1.0.conv1", "layer1.1.conv1", "layer1.2.conv1"]+
    ["layer2.0.conv1", "layer2.1.conv1", "layer2.2.conv1"]+
    ["layer3.0.conv1", "layer3.1.conv1", "layer3.2.conv1"]+
    ["layer4.0.conv1", "layer4.1.conv1", "layer4.2.conv1"]
)


# plot grid of heatmaps
fig, ax = plt.subplots(4, 3, figsize=(20, 20))
for i, layer_name in enumerate(layer_names):
    heatmap = grads[layer_name].squeeze(0).cpu().detach().mean(dim=0).numpy()
    heatmap = np.maximum(heatmap, 0)
    ax[i // 3, i % 3].imshow(heatmap, cmap="hot", interpolation="nearest")
    ax[i // 3, i % 3].set_title(layer_name)
    ax[i // 3, i % 3].axis("off")
plt.tight_layout()